In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd


In [3]:
KEYWORD = "폴로"

CSV_PATH = Path(f"../data/csv/daangn_{KEYWORD}.csv")
IMG_DIR = Path(f"../data/images/{KEYWORD}")

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".jfif"}

# -----------------------------
# load df
# -----------------------------
df = pd.read_csv(CSV_PATH)

df_ids = set(df["id"].astype(str).str.strip())

print(f"df rows: {len(df_ids)}")

# -----------------------------
# scan image directory
# -----------------------------
img_paths = [p for p in IMG_DIR.rglob("*") if p.is_file()]
img_paths = [p for p in img_paths if p.suffix.lower() in EXTS]

print(f"image files: {len(img_paths)}")

img_ids = set(p.stem.strip() for p in img_paths)

# -----------------------------
# compare
# -----------------------------
missing_images = df_ids - img_ids
extra_images = img_ids - df_ids
common = df_ids & img_ids

print("\n===== SUMMARY =====")
print("df ids:", len(df_ids))
print("image ids:", len(img_ids))
print("matched:", len(common))
print("missing images (df only):", len(missing_images))
print("extra images (image only):", len(extra_images))

# -----------------------------
# sample outputs
# -----------------------------
print("\n===== SAMPLE: df에 있는데 이미지 없는 id =====")
print(list(missing_images)[:20])

print("\n===== SAMPLE: 이미지 있는데 df에 없는 id =====")
print(list(extra_images)[:20])

# -----------------------------
# extension distribution
# -----------------------------
ext_counter = Counter(p.suffix.lower() for p in img_paths)

print("\n===== EXTENSION DISTRIBUTION =====")
for ext, cnt in ext_counter.items():
    print(ext, cnt)

# -----------------------------
# duplicate stems
# -----------------------------
stem_counter = Counter(p.stem for p in img_paths)

dups = [k for k, v in stem_counter.items() if v > 1]

print("\n===== DUPLICATE STEMS =====")
print(len(dups))

if dups:
    print(dups[:20])

df rows: 8970
image files: 8959

===== SUMMARY =====
df ids: 8970
image ids: 8959
matched: 8959
missing images (df only): 11
extra images (image only): 0

===== SAMPLE: df에 있는데 이미지 없는 id =====
['65vkmk2m4fdc', '7ybag7gf26i1', 'hoc9he4wthhk', 'a37w7tgrgn5r', '9qh9kpu91j85', 'oiketcazsrux', 'oo1q9tmv86ss', 'szybwp3diaqs', 'f1txi5nvi4np', 'a5poy4cy4mwc', '8ej6jy43xmdz']

===== SAMPLE: 이미지 있는데 df에 없는 id =====
[]

===== EXTENSION DISTRIBUTION =====
.jpg 1885
.webp 7074

===== DUPLICATE STEMS =====
0


In [5]:
from pathlib import Path

KEYWORD="폴로"
IMG_DIR=Path(f"../data/images/{KEYWORD}")
EXTS={".jpg",".jpeg",".png",".webp",".jfif"}

root_cnt = sum(1 for p in IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXTS)
all_cnt  = sum(1 for p in IMG_DIR.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)

print("root files:", root_cnt)
print("all files (recursive):", all_cnt)

root files: 8959
all files (recursive): 8959


In [6]:
from pathlib import Path
import pandas as pd
import unicodedata

KEYWORD = "폴로"
CSV_PATH = Path(f"../data/csv/daangn_{KEYWORD}.csv")
IMG_DIR  = Path(f"../data/images/{KEYWORD}")
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".jfif"}

def exists_for_id(item_id: str) -> bool:
    for ext in EXTS:
        if (IMG_DIR / f"{item_id}{ext}").exists():
            return True
        if (IMG_DIR / f"{item_id}{ext.upper()}").exists():
            return True
    return False

def show_weird(s: str) -> str:
    # show unicode categories for each char (helps spot BOM/ZWSP/NBSP)
    parts = []
    for ch in s:
        name = unicodedata.name(ch, "UNKNOWN")
        cat = unicodedata.category(ch)
        parts.append(f"{ch}({cat},{name})")
    return " ".join(parts)

df = pd.read_csv(CSV_PATH)
df["id"] = df["id"].astype(str)

missing = []
for raw in df["id"]:
    item_id = raw.strip()
    if not exists_for_id(item_id):
        missing.append(raw)

print("missing count (as code currently checks):", len(missing))

# print first 20 with repr + unicode breakdown
for s in missing[:20]:
    print("\nRAW repr:", repr(s))
    print("STRIP repr:", repr(s.strip()))
    print("unicode:", show_weird(s))

missing count (as code currently checks): 11

RAW repr: 'a37w7tgrgn5r'
STRIP repr: 'a37w7tgrgn5r'
unicode: a(Ll,LATIN SMALL LETTER A) 3(Nd,DIGIT THREE) 7(Nd,DIGIT SEVEN) w(Ll,LATIN SMALL LETTER W) 7(Nd,DIGIT SEVEN) t(Ll,LATIN SMALL LETTER T) g(Ll,LATIN SMALL LETTER G) r(Ll,LATIN SMALL LETTER R) g(Ll,LATIN SMALL LETTER G) n(Ll,LATIN SMALL LETTER N) 5(Nd,DIGIT FIVE) r(Ll,LATIN SMALL LETTER R)

RAW repr: 'oo1q9tmv86ss'
STRIP repr: 'oo1q9tmv86ss'
unicode: o(Ll,LATIN SMALL LETTER O) o(Ll,LATIN SMALL LETTER O) 1(Nd,DIGIT ONE) q(Ll,LATIN SMALL LETTER Q) 9(Nd,DIGIT NINE) t(Ll,LATIN SMALL LETTER T) m(Ll,LATIN SMALL LETTER M) v(Ll,LATIN SMALL LETTER V) 8(Nd,DIGIT EIGHT) 6(Nd,DIGIT SIX) s(Ll,LATIN SMALL LETTER S) s(Ll,LATIN SMALL LETTER S)

RAW repr: 'oiketcazsrux'
STRIP repr: 'oiketcazsrux'
unicode: o(Ll,LATIN SMALL LETTER O) i(Ll,LATIN SMALL LETTER I) k(Ll,LATIN SMALL LETTER K) e(Ll,LATIN SMALL LETTER E) t(Ll,LATIN SMALL LETTER T) c(Ll,LATIN SMALL LETTER C) a(Ll,LATIN SMALL LETTER A) z(Ll,LATIN